# Phase 1 - Arrondissement Data Exploration

Notebook d'exploration de la table `master_arrondissements` dans la voie pedagogique officielle du projet.


## 1. Setup & Imports


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import build_pedagogical_master_table

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Data Loading


In [ ]:
master_arr = build_pedagogical_master_table()
master_arr.head()


## 3. Data Quality Check

Verification des types, des valeurs manquantes et de la cardinalite des 20 arrondissements.


In [ ]:
pd.DataFrame({
    "dtype": master_arr.drop(columns="geometry").dtypes.astype(str),
    "na_count": master_arr.drop(columns="geometry").isna().sum(),
})


In [ ]:
master_arr[["arrondissement_code", "arrondissement_name"]].sort_values("arrondissement_code")


## 4. Descriptive Statistics

Les variables `X1..X5` et la cible `Y` sont analysees a l'echelle arrondissement.


In [ ]:
analysis_columns = [
    "y_bin_count",
    "x1_population",
    "x2_commerce_restaurant_count",
    "x3_transport_station_count",
    "x4_green_area_m2",
    "x5_road_length_km",
    "x6_terrasse_surface_m2",
    "x7_school_count",
]
master_arr[analysis_columns].describe().T


In [ ]:
plot_df = master_arr.sort_values("y_bin_count", ascending=False)
ax = plot_df.plot(
    x="arrondissement_name",
    y="y_bin_count",
    kind="bar",
    figsize=(12, 5),
    legend=False,
    color="#1f77b4",
)
ax.set_title("Observed street bins by arrondissement")
ax.set_xlabel("")
ax.set_ylabel("OSM waste basket count")
plt.xticks(rotation=75)
plt.show()


## 5. Geographic Visualization


In [ ]:
ax = master_arr.plot(
    column="y_bin_count",
    cmap="YlOrRd",
    figsize=(8, 8),
    legend=True,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_title("Observed street bins by arrondissement")
ax.set_axis_off()
plt.show()


## 6. Correlation Analysis


In [ ]:
master_arr[analysis_columns].corr(numeric_only=True)


## 7. Scatterplots Y vs X1..X5

Chaque variable explicative est tracee contre la cible y_bin_count.

In [ ]:
feature_labels = [
    ("x1_population",                "X1 Population"),
    ("x2_commerce_restaurant_count", "X2 Commerces/Restaurants"),
    ("x3_transport_station_count",   "X3 Stations de transport"),
    ("x4_green_area_m2",             "X4 Surface verte (m2)"),
    ("x5_road_length_km",            "X5 Longueur routes (km)"),
    ("x6_terrasse_surface_m2",       "X6 Surface terrasses (m2)"),
    ("x7_school_count",              "X7 Etablissements scolaires"),
]

fig, axes = plt.subplots(1, 7, figsize=(28, 4))
for ax, (col, label) in zip(axes, feature_labels):
    ax.scatter(master_arr[col], master_arr["y_bin_count"],
               color="steelblue", edgecolors="white", linewidths=0.5)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Y (bin count)" if ax is axes[0] else "")
    ax.set_title("Y vs " + label.split()[0], fontsize=10)
fig.suptitle("Scatterplots : Y (bin count) vs X1..X7", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

## 8. Matrice de correlation

Heatmap lisible des correlations entre Y et X1..X5.

In [ ]:
import numpy as np

corr = master_arr[analysis_columns].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.values, cmap="RdBu", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label="Correlation de Pearson")

labels = ["Y bin count", "X1 Pop", "X2 Commerce", "X3 Transport",
          "X4 Vert", "X5 Routes", "X6 Terrasses", "X7 Ecoles"]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="black" if abs(val) < 0.7 else "white")

ax.set_title("Matrice de correlation - Y et X1..X7", fontsize=12)
fig.tight_layout()
plt.show()

print("Correlations avec Y (y_bin_count) :")
print(corr["y_bin_count"].drop("y_bin_count").sort_values(ascending=False).round(3))

## 9. Tableau des valeurs manquantes

Verification explicite de l'absence de NaN dans les colonnes de modelisation.

In [ ]:
missing_df = pd.DataFrame({
    "colonne": analysis_columns,
    "na_count": [master_arr[col].isna().sum() for col in analysis_columns],
    "na_pct": [master_arr[col].isna().mean() * 100 for col in analysis_columns],
    "dtype": [str(master_arr[col].dtype) for col in analysis_columns],
})
print("Valeurs manquantes dans les colonnes de modelisation :")
missing_df

## 10. Feature engineering supplementaire ?

Conclusion EDA : aucun feature engineering supplementaire n'est necessaire pour la phase 2.

- Les 5 variables sont sur des echelles tres differentes -> StandardScaler dans le pipeline MLP suffit.
- Pas de bi-modalite ou asymetrie extreme necessitant une transformation log.
- Avec 20 observations, ajouter des features derivees augmenterait le sur-apprentissage.
- La correlation entre X1, X2, X5 est moderee a forte : la regularisation L2 du MLP (alpha=0.001) gere la colinearite.

## 11. Synthese et passage a la phase 2

- La table master_arrondissements contient exactement 20 observations, une par arrondissement.
- La cible y_bin_count et les variables X1..X5 ne contiennent aucune valeur manquante.
- Les correlations montrent des relations moderees entre Y et les features.
- Aucun feature engineering supplementaire n'est ajoute : StandardScaler dans le pipeline MLP est suffisant.
- La phase 2 demarre a partir de cette table via run_phase2_neural_network_pipeline() dans 02_modeling.ipynb.